In [2]:
text = "the cat sat on the mat. the cat ate the rat. the rat sat on the cat."

In [3]:
len(text)

68

In [4]:
chars = sorted(text)

In [5]:
print(chars)

[' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', '.', '.', '.', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'c', 'c', 'c', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'h', 'h', 'h', 'h', 'h', 'h', 'm', 'n', 'n', 'o', 'o', 'r', 'r', 's', 's', 't', 't', 't', 't', 't', 't', 't', 't', 't', 't', 't', 't', 't', 't', 't']


In [6]:
chars = sorted(set(text))

In [7]:
print(chars)

[' ', '.', 'a', 'c', 'e', 'h', 'm', 'n', 'o', 'r', 's', 't']


In [8]:
vocab_size = len(chars)

In [9]:
print(vocab_size)

12


In [10]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i : ch for i, ch in enumerate(chars)}

encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [11]:
encode(text)

[11,
 5,
 4,
 0,
 3,
 2,
 11,
 0,
 10,
 2,
 11,
 0,
 8,
 7,
 0,
 11,
 5,
 4,
 0,
 6,
 2,
 11,
 1,
 0,
 11,
 5,
 4,
 0,
 3,
 2,
 11,
 0,
 2,
 11,
 4,
 0,
 11,
 5,
 4,
 0,
 9,
 2,
 11,
 1,
 0,
 11,
 5,
 4,
 0,
 9,
 2,
 11,
 0,
 10,
 2,
 11,
 0,
 8,
 7,
 0,
 11,
 5,
 4,
 0,
 3,
 2,
 11,
 1]

In [12]:
import torch
text= torch.tensor(encode(text), dtype=torch.long)

In [13]:
n = int(0.9 * len(text)) # 90 % partition
train_data = text[:n] # 90% training data , rest validation
val_data = text[n:] # 10% validation data

In [14]:
batch_size = 4
block_size = 8

In [15]:

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, )) # this gives you {batch_size} random numbers from 0 to data_length - block_size
    x = torch.stack([data[i : i + block_size] for i in ix]) # 
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print('----')

# basically kinda looping through our stack of 4 * 8
for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension 
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"When input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[10,  2, 11,  0,  8,  7,  0, 11],
        [ 5,  4,  0,  9,  2, 11,  0, 10],
        [ 5,  4,  0,  6,  2, 11,  1,  0],
        [ 0,  9,  2, 11,  1,  0, 11,  5]])
targets:
torch.Size([4, 8])
tensor([[ 2, 11,  0,  8,  7,  0, 11,  5],
        [ 4,  0,  9,  2, 11,  0, 10,  2],
        [ 4,  0,  6,  2, 11,  1,  0, 11],
        [ 9,  2, 11,  1,  0, 11,  5,  4]])
----
When input is [10] the target: 2
When input is [10, 2] the target: 11
When input is [10, 2, 11] the target: 0
When input is [10, 2, 11, 0] the target: 8
When input is [10, 2, 11, 0, 8] the target: 7
When input is [10, 2, 11, 0, 8, 7] the target: 0
When input is [10, 2, 11, 0, 8, 7, 0] the target: 11
When input is [10, 2, 11, 0, 8, 7, 0, 11] the target: 5
When input is [5] the target: 4
When input is [5, 4] the target: 0
When input is [5, 4, 0] the target: 9
When input is [5, 4, 0, 9] the target: 2
When input is [5, 4, 0, 9, 2] the target: 11
When input is [5, 4, 0, 9, 2, 11] the target: 0
When i

In [43]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Below code think it as 2d matrix of vocab_size * vocab_size (in our case - 12 * 12)
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # our entire model (almost all magic happens here) 
        

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # logits = raw scores for each possible next token 
          
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) 
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) 
        
        return logits, loss # (B, T, C)

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)  
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim = -1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sample index to the running sequence 
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)
        return idx
        
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb) # # calling the model 
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


tensor([[[ 0.1808, -0.0700, -0.3596, -0.9152,  0.6258,  0.0255,  0.9545,
           0.0643,  0.3612,  1.1679, -1.3499, -0.5102],
         [-1.6669, -1.3651, -0.1655,  0.9623,  0.0315, -0.7419, -0.2978,
           0.0172, -0.1772, -0.1334,  0.2940,  1.3850],
         [-1.5935, -1.2706,  0.6903, -0.1961,  0.3449, -0.3419,  0.4759,
          -0.7663, -0.4190, -0.4370, -1.0012, -0.4094],
         [ 0.1808, -0.0700, -0.3596, -0.9152,  0.6258,  0.0255,  0.9545,
           0.0643,  0.3612,  1.1679, -1.3499, -0.5102],
         [ 1.3471,  1.6910, -0.1244, -1.6824, -0.0266,  0.0740,  1.0517,
           0.6779,  0.3067, -0.7472,  0.7435,  0.8877],
         [ 0.7789,  1.5333,  1.6097, -0.4032, -0.8345,  0.5978, -0.0514,
          -0.0646, -0.4970,  0.4658, -0.2573, -1.0673],
         [ 1.6455, -0.8030,  1.3514, -0.2759, -1.5108,  2.1048,  2.7630,
          -1.7465,  1.4516, -1.5103,  0.8212, -0.2115],
         [ 0.1808, -0.0700, -0.3596, -0.9152,  0.6258,  0.0255,  0.9545,
           0.0643,  0.36

In [39]:
# create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)  # changes the model's parameters

In [42]:
batch_size = 4
for step in range(10):
    # sample a batch of data
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()  # figures out how the parameters contributed to the error
    optimizer.step() # actually changes those parameters

    print(loss.item())

this is B 4
this is T 8
this is C 12
targets:  tensor([11,  5,  4,  0,  9,  2, 11,  1,  2, 11,  4,  0, 11,  5,  4,  0,  9,  2,
        11,  0, 10,  2, 11,  0, 11,  0, 10,  2, 11,  0,  8,  7])
2.7859678268432617
this is B 4
this is T 8
this is C 12
targets:  tensor([ 0, 10,  2, 11,  0,  8,  7,  0,  4,  0,  6,  2, 11,  1,  0, 11,  4,  0,
        11,  5,  4,  0,  9,  2, 11,  0,  2, 11,  4,  0, 11,  5])
2.786259651184082
this is B 4
this is T 8
this is C 12
targets:  tensor([ 0, 10,  2, 11,  0,  8,  7,  0, 11,  0,  8,  7,  0, 11,  5,  4,  4,  0,
         9,  2, 11,  0, 10,  2, 11,  0,  8,  7,  0, 11,  5,  4])
2.896900177001953
this is B 4
this is T 8
this is C 12
targets:  tensor([ 0,  3,  2, 11,  0,  2, 11,  4,  0,  2, 11,  4,  0, 11,  5,  4, 11,  5,
         4,  0,  6,  2, 11,  1, 11,  5,  4,  0,  3,  2, 11,  0])
2.8937673568725586
this is B 4
this is T 8
this is C 12
targets:  tensor([11,  1,  0, 11,  5,  4,  0,  3,  9,  2, 11,  0, 10,  2, 11,  0, 11,  5,
         4,  0,  9,  2, 11,  0,

In [31]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))

 rathe t. the rathe sathe mat at rat rathe cate the t t at on rathe t rat at mathe t cathe t on sathe cat mathe sathe te sathe sathe sate t. t cathe on t t cat sat rathe t. cat. t rate cat rathe sat t. t t cate on at. t athe t rat cat t athe mat cat t rate on t. at t t t cathe rat. athe the mathe on rat the sathe t. t sat cathe t the te cathe cat cate t rat. cathe rathe on t the cat t. the on rat cathe cathe sat cathe t sat. on rate t. t rat athe rat. the rate t. cathe t rathe cat t rate cat sathe rat. at. cathe sate sat. on mathe rat at cat cat rathe cat the the cathe sat rathe sat rat. rat athe at t rathe sathe mathe t t athe athe t rat. sat rat. rathe the rat sat the the cat at. at cat. mathe sat t rat mat te the sat sathe sathe t sat. on sathe t mat sat. rat. on t mate t. on t. the on the the the t. rat. rathe cat the the cat cat. cathe athe the sat on t the cat sathe rathe cat cathe on mat. the te t. rathe rat te the on rate sat mat sat cathe rat the rat mathe on rat. on the mate 